# Stanford PHI NER Evaluation on MACCROBAT

This notebook performs a scoped token-level evaluation of `StanfordAIMI/stanford-deidentifier-base` on all 400 MACCROBAT documents.

MACCROBAT is not a PHI de-identification dataset. Only `Date -> DATE` and the coarse mapping `Nonbiological_location -> HOSPITAL` are evaluated. MACCROBAT has no compatible gold labels for `HCW`, `PATIENT`, `ID`, `PHONE`, or `VENDOR`, so those categories are excluded from scoped precision, recall, and F1.

## Load the Model

In [1]:
from __future__ import annotations

import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)

MODEL_NAME = "StanfordAIMI/stanford-deidentifier-base"
IGNORE_LABEL_ID = -100

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)

print(f"Loaded {MODEL_NAME}.")
print("Evaluation device:", "cuda" if torch.cuda.is_available() else "cpu")
print("Model labels:", model.config.id2label)

python(30098) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(30099) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: StanfordAIMI/stanford-deidentifier-base
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded StanfordAIMI/stanford-deidentifier-base.
Evaluation device: cpu
Model labels: {0: 'O', 1: 'VENDOR', 2: 'DATE', 3: 'HCW', 4: 'HOSPITAL', 5: 'ID', 6: 'PATIENT', 7: 'PHONE'}


## Load MACCROBAT

In [2]:
maccrobat_evaluation = load_dataset(
    "ktgiahieu/maccrobat2018_2020",
    split="train",
)

print("Evaluation documents:", len(maccrobat_evaluation))
print("Dataset columns:", maccrobat_evaluation.column_names)

Evaluation documents: 400
Dataset columns: ['tokens', 'tags']


## Define the Scoped Label Mapping

In [3]:
GOLD_ENTITY_MAP = {
    "Date": "DATE",
    "Nonbiological_location": "HOSPITAL",
}
TARGET_LABELS = ("DATE", "HOSPITAL")

missing_model_labels = sorted(set(TARGET_LABELS) - set(model.config.label2id))
print("Evaluated labels:", TARGET_LABELS)
print("Excluded model labels:", sorted(set(model.config.label2id) - {"O", *TARGET_LABELS}))
print("Missing required labels:", missing_model_labels)

if missing_model_labels:
    raise ValueError("The model is missing labels required by the scoped mapping.")

Evaluated labels: ('DATE', 'HOSPITAL')
Excluded model labels: ['HCW', 'ID', 'PATIENT', 'PHONE', 'VENDOR']
Missing required labels: []


## Tokenize and Align Labels

The PHI model uses flat token classes rather than BIO-prefixed labels. MACCROBAT's `B-` and `I-` prefixes are therefore removed during mapping. Only the first subtoken of each original word is scored.

In [5]:
def map_gold_label(label: str) -> str:
    if label == "O":
        return "O"

    _, entity_type = label.split("-", maxsplit=1)
    return GOLD_ENTITY_MAP.get(entity_type, "O")


def tokenize_and_align_maccrobat(examples):
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=512,
        stride=0,
        return_overflowing_tokens=True,
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    aligned_labels = []

    for chunk_index, sample_index in enumerate(sample_mapping):
        word_ids = tokenized.word_ids(batch_index=chunk_index)
        document_tags = examples["tags"][sample_index]
        chunk_labels = []
        previous_word_id = None

        for word_id in word_ids:
            if word_id is None or word_id == previous_word_id:
                chunk_labels.append(IGNORE_LABEL_ID)
            else:
                mapped_label = map_gold_label(document_tags[word_id])
                chunk_labels.append(model.config.label2id[mapped_label])
            previous_word_id = word_id

        aligned_labels.append(chunk_labels)

    tokenized["labels"] = aligned_labels
    return tokenized


tokenized_evaluation = maccrobat_evaluation.map(
    tokenize_and_align_maccrobat,
    batched=True,
    remove_columns=maccrobat_evaluation.column_names,
    desc="Tokenizing MACCROBAT PHI evaluation set",
)

print("Evaluation documents:", len(maccrobat_evaluation))
print("Tokenized chunks:", len(tokenized_evaluation))

Evaluation documents: 400
Tokenized chunks: 650


## Calculate Scoped Token Metrics

Precision, recall, and F1 are micro-averaged over `DATE` and `HOSPITAL`. Accuracy covers all scored tokens and is likely dominated by `O`.

In [6]:
def safe_divide(numerator: int, denominator: int) -> float:
    return numerator / denominator if denominator else 0.0


def compute_phi_metrics(evaluation_result):
    logits, label_ids = evaluation_result
    predicted_ids = np.argmax(logits, axis=-1)
    valid_mask = label_ids != IGNORE_LABEL_ID
    flat_predictions = predicted_ids[valid_mask]
    flat_references = label_ids[valid_mask]

    total_true_positives = 0
    total_false_positives = 0
    total_false_negatives = 0
    metrics = {}

    for label in TARGET_LABELS:
        label_id = model.config.label2id[label]
        true_positives = int(np.sum((flat_predictions == label_id) & (flat_references == label_id)))
        false_positives = int(np.sum((flat_predictions == label_id) & (flat_references != label_id)))
        false_negatives = int(np.sum((flat_predictions != label_id) & (flat_references == label_id)))

        precision = safe_divide(true_positives, true_positives + false_positives)
        recall = safe_divide(true_positives, true_positives + false_negatives)
        metrics[f"{label.lower()}_precision"] = precision
        metrics[f"{label.lower()}_recall"] = recall
        metrics[f"{label.lower()}_f1"] = safe_divide(2 * precision * recall, precision + recall)

        total_true_positives += true_positives
        total_false_positives += false_positives
        total_false_negatives += false_negatives

    micro_precision = safe_divide(
        total_true_positives, total_true_positives + total_false_positives
    )
    micro_recall = safe_divide(
        total_true_positives, total_true_positives + total_false_negatives
    )
    metrics.update(
        {
            "precision": micro_precision,
            "recall": micro_recall,
            "f1": safe_divide(
                2 * micro_precision * micro_recall, micro_precision + micro_recall
            ),
            "accuracy": float(np.mean(flat_predictions == flat_references)),
        }
    )
    return metrics

In [7]:
evaluation_arguments = TrainingArguments(
    output_dir="outputs/stanford-phi-maccrobat-evaluation",
    per_device_eval_batch_size=8,
    report_to="none",
)

evaluator = Trainer(
    model=model,
    args=evaluation_arguments,
    eval_dataset=tokenized_evaluation,
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_phi_metrics,
)

evaluation_metrics = evaluator.evaluate()
evaluation_metrics

/opt/anaconda3/envs/dl-lab/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step,Date Precision,Date Recall,Date F1,Hospital Precision,Hospital Recall,Hospital F1,Precision,Recall,F1,Accuracy
No log,0.204112,0,0.767677,0.150993,0.252352,0.483333,0.217636,0.300129,0.640596,0.168380,0.266667,0.980900


{'eval_loss': 0.20411249995231628,
 'eval_date_precision': 0.7676767676767676,
 'eval_date_recall': 0.1509933774834437,
 'eval_date_f1': 0.2523519645821804,
 'eval_hospital_precision': 0.48333333333333334,
 'eval_hospital_recall': 0.2176360225140713,
 'eval_hospital_f1': 0.3001293661060802,
 'eval_precision': 0.6405959031657356,
 'eval_recall': 0.16837983357807146,
 'eval_f1': 0.26666666666666666,
 'eval_accuracy': 0.98089956272603}